[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/sodacore-certified/notebooks/day-01-soda-core-intro.ipynb#scrollTo=a1b2c301)

---
# Day 1 · Soda Core Architecture — Checks, SodaCL, and Scan Execution
**certified-journeys / sodacore-certified** · Day 1 · Foundations

> **Goal for today:** Understand where Soda Core fits in the data quality landscape, write your first SodaCL checks file, and execute a scan programmatically using the Python API — reading and interpreting PASS, FAIL, and WARN results.

In [ ]:
%pip install -q soda-core-duckdb

## Step 1 · Soda Core in the Data Quality Landscape

Soda Core is an **open-source data quality framework** that lets you define checks in YAML (SodaCL) and execute them against any supported data source. It sits between two extremes:

| Tool | Approach | Where checks live | Best for |
|------|----------|-------------------|----------|
| **Great Expectations** | Python-first, expectation suites | Python files / JSON stores | Data scientists, Python-heavy teams |
| **Soda Core** | YAML-first (SodaCL), Python API | `.yml` check files | Data engineers, dbt users, cross-team |
| **dbt tests** | SQL macros in YAML | `schema.yml` in dbt project | dbt-centric workflows |
| **Custom SQL asserts** | Raw SQL | Migration scripts / CI | Small teams, existing SQL skills |

**Why Soda Core?**
- Checks are human-readable YAML — reviewable in PRs by non-engineers.
- The Python `Scan()` API integrates into any pipeline (Airflow, Prefect, GitHub Actions).
- One framework covers 20+ data sources: DuckDB, PostgreSQL, Snowflake, BigQuery, Spark, and more.
- Results are structured objects — you can route failures to Slack, PagerDuty, or a custom dashboard.

In [ ]:
# Verify the install — print soda-core version
import importlib.metadata

try:
    version = importlib.metadata.version("soda-core-duckdb")
    print(f"soda-core-duckdb version: {version}")
except importlib.metadata.PackageNotFoundError:
    print("Package not found — re-run the install cell above")

# Also confirm the Scan class is importable
from soda.scan import Scan
print("Scan class imported successfully:", Scan)

### What just happened?
- `soda-core-duckdb` installs the Soda Core framework **plus** the DuckDB connector in one package.
- **`from soda.scan import Scan`** is the single import you need to run scans programmatically.
- In production you would install `soda-core-snowflake`, `soda-core-postgres`, etc. for each data source.
- The DuckDB connector is ideal for local development because it requires no running server.

## Step 2 · SodaCL — The Checks Language

**SodaCL (Soda Checks Language)** is a YAML dialect for expressing data quality rules. A checks file has one or more `checks for <table>` blocks:

```yaml
checks for orders:
  - row_count > 0
  - missing_count(customer_id) = 0
  - duplicate_count(order_id) = 0

checks for customers:
  - row_count > 0
  - missing_percent(email) < 5%
```

**Anatomy of a check:**

| Part | Example | Meaning |
|------|---------|--------|
| Metric | `row_count` | What to measure |
| Column (optional) | `(customer_id)` | Which column to measure |
| Operator | `=`, `>`, `<`, `>=`, `<=` | Comparison type |
| Threshold | `0`, `100`, `5%` | Pass/fail boundary |

**Warn vs Fail thresholds** — add a `warn` key to emit a WARNING instead of FAIL for softer violations:

```yaml
checks for orders:
  - row_count:
      fail: when < 100
      warn: when < 500
```

In [ ]:
# Demonstrate SodaCL YAML structure using Python's yaml module
import yaml

# A checks file is just a Python dict — helpful to understand the structure
checks_structure = {
    "checks for orders": [
        "row_count > 0",
        "missing_count(customer_id) = 0",
        "duplicate_count(order_id) = 0",
        {
            "row_count": {
                "fail": "when < 100",
                "warn": "when < 500"
            }
        }
    ],
    "checks for customers": [
        "row_count > 0",
        "missing_percent(email) < 5%"
    ]
}

print("=== SodaCL YAML structure ===")
print(yaml.dump(checks_structure, default_flow_style=False, sort_keys=False))

### What just happened?
- SodaCL checks are plain YAML — each item in a `checks for <table>` block is either a **string expression** or a **dict with fail/warn thresholds**.
- **String form** (`"row_count > 0"`) is the concise shorthand — Soda parses the operator and threshold automatically.
- **Dict form** lets you set independent `fail` and `warn` thresholds for the same metric.
- `missing_percent` accepts a `%` suffix — Soda interprets it as a percentage (0–100 scale).

## Step 3 · Configuration — Connecting a Data Source

Soda uses a separate `configuration.yml` to define data source connections. This keeps credentials out of the checks files and lets you reuse the same checks against different environments.

**DuckDB configuration.yml:**

```yaml
data_sources:
  my_duckdb:
    type: duckdb
    path: /tmp/my_database.duckdb
```

**Scan lifecycle — what happens when you call `scan.execute()`:**

```
1. Scan reads configuration.yml → opens connection to data source
2. Scan reads checks.yml       → parses each check into a metric query
3. Soda generates SQL          → one query per metric per table
4. SQL runs on the data source → raw values returned
5. Soda evaluates thresholds   → PASS / FAIL / WARN per check
6. Results collected           → accessible via scan.get_logs_text(),
                                  scan.get_checks_fail(), etc.
```

Key insight: **Soda pushes computation to the data source** — it never moves data out. All metric queries run as SQL on your warehouse.

In [ ]:
import duckdb
import tempfile
import pathlib
import os

# --- 1. Create synthetic data in a temp DuckDB file ---
tmpdir = pathlib.Path(tempfile.mkdtemp())
db_path = str(tmpdir / "warehouse.duckdb")

conn = duckdb.connect(db_path)
conn.execute("""
    CREATE TABLE orders AS
    SELECT * FROM (VALUES
        (1,  'Alice',   120.50, '2024-01-01'),
        (2,  'Bob',     250.00, '2024-01-02'),
        (3,  'Charlie', 89.99,  '2024-01-03'),
        (4,  'Diana',   340.00, '2024-01-04'),
        (5,  'Eve',     175.25, '2024-01-05')
    ) t(order_id, customer, amount, order_date)
""")
conn.close()  # close so Soda can open it

# --- 2. Write configuration.yml pointing at our temp DuckDB file ---
config_yml = f"""
data_sources:
  warehouse:
    type: duckdb
    path: "{db_path}"
"""

# --- 3. Write a simple checks.yml ---
checks_yml = """
checks for orders:
  - row_count > 0
"""

config_path = tmpdir / "configuration.yml"
checks_path = tmpdir / "checks.yml"
config_path.write_text(config_yml)
checks_path.write_text(checks_yml)

print(f"DuckDB file : {db_path}")
print(f"Config path : {config_path}")
print(f"Checks path : {checks_path}")
print("\n--- configuration.yml ---")
print(config_yml.strip())
print("\n--- checks.yml ---")
print(checks_yml.strip())

### What just happened?
- We created a real DuckDB file on disk — **not `:memory:`** — because Soda needs to open its own connection.
- We closed the DuckDB connection before handing the path to Soda — DuckDB allows multi-process reads but only one write connection.
- `configuration.yml` references the **data source name** (`warehouse`) that checks files will target.
- The checks file uses the simple string form: `row_count > 0` means FAIL if the table has zero rows.

## Step 4 · Running Your First Scan with the Python API

The `Scan` class is the main entry point for programmatic scan execution. The minimal workflow:

```python
from soda.scan import Scan

scan = Scan()
scan.set_data_source_name("my_data_source")   # must match configuration.yml key
scan.add_configuration_yaml_file("configuration.yml")
scan.add_sodacl_yaml_file("checks.yml")
scan.execute()

print(scan.get_logs_text())
```

After `execute()`, the scan object holds all results — you can inspect them without printing raw logs.

In [ ]:
from soda.scan import Scan

# Run the scan against our DuckDB warehouse
scan = Scan()
scan.set_data_source_name("warehouse")             # matches the key in configuration.yml
scan.add_configuration_yaml_file(str(config_path))
scan.add_sodacl_yaml_file(str(checks_path))
scan.execute()

# Print the full scan log
print(scan.get_logs_text())

### What just happened?
- `scan.set_data_source_name("warehouse")` tells Soda which connection block in `configuration.yml` to use.
- `scan.execute()` connects to DuckDB, generates a `SELECT COUNT(*)` query, and evaluates `row_count > 0`.
- The log shows each check with its **outcome** (PASS/FAIL/WARN) and the **measured value** (e.g. `row_count = 5`).
- **`get_logs_text()`** returns a human-readable summary — useful for CI pipeline output.

## Step 5 · Interpreting Scan Results — PASS, FAIL, WARN

Soda checks produce one of three outcomes:

| Outcome | Meaning | Default exit behaviour |
|---------|---------|------------------------|
| **PASS** | Metric satisfies the threshold | Scan succeeds (exit 0) |
| **WARN** | Metric exceeds the warn threshold but not the fail threshold | Scan completes with warnings (exit 0) |
| **FAIL** | Metric violates the fail threshold | Scan fails (non-zero exit in CLI) |

With the Python API you inspect results programmatically:
- `scan.has_check_fails()` → `True` if any check FAILed
- `scan.has_check_warns()` → `True` if any check WARNed
- `scan.get_checks_fail()` → list of failed check objects
- `scan.get_checks_warn()` → list of warned check objects
- `scan.get_all_checks_text()` → formatted string with all check outcomes

In [ ]:
# --- Expand checks.yml with PASS, WARN, and FAIL scenarios ---
checks_multi_yml = """
checks for orders:
  - row_count > 0
  - row_count:
      name: row count warn/fail thresholds
      warn: when < 3
      fail: when < 1
  - missing_count(customer) = 0
  - duplicate_count(order_id) = 0
"""
checks_path2 = tmpdir / "checks_multi.yml"
checks_path2.write_text(checks_multi_yml)

scan2 = Scan()
scan2.set_data_source_name("warehouse")
scan2.add_configuration_yaml_file(str(config_path))
scan2.add_sodacl_yaml_file(str(checks_path2))
scan2.execute()

# Inspect results programmatically
print("=== Scan outcome summary ===")
print(f"  Any FAILs?  {scan2.has_check_fails()}")
print(f"  Any WARNs?  {scan2.has_check_warns()}")

if scan2.has_check_fails():
    print("\nFailed checks:")
    for chk in scan2.get_checks_fail():
        print(f"  FAIL  {chk}")
else:
    print("\nNo check failures.")

if scan2.has_check_warns():
    print("\nWarned checks:")
    for chk in scan2.get_checks_warn():
        print(f"  WARN  {chk}")
else:
    print("No warnings.")

print("\n=== All checks text ===")
print(scan2.get_logs_text())

### What just happened?
- `scan.has_check_fails()` and `scan.has_check_warns()` are the two booleans you test in CI pipeline guards.
- The **named check** (`name: row count warn/fail thresholds`) makes scan output more readable — names appear in logs and dashboards.
- `duplicate_count(order_id) = 0` generates a `COUNT(*) - COUNT(DISTINCT order_id)` style query — Soda handles the SQL.
- With 5 rows all having unique `order_id` values and no nulls in `customer`, all checks should PASS on our clean dataset.

## Step 6 · Soda Core Architecture — How Resolution Works

Understanding how Soda resolves data sources, checks, and configuration helps you debug connection errors and organise large projects.

**Resolution order:**

```
scan.set_data_source_name("warehouse")
           │
           ▼
configuration.yml  ←  scan.add_configuration_yaml_file(path)
  data_sources:
    warehouse:         ← name matched here
      type: duckdb
      path: /tmp/...
           │
           ▼
checks.yml  ←  scan.add_sodacl_yaml_file(path)
  checks for orders:  ← table name resolved against the data source
    - row_count > 0
           │
           ▼
SQL generated:  SELECT COUNT(*) FROM orders
           │
           ▼
Results stored in Scan object
```

**Multi-file scans** — you can call `add_sodacl_yaml_file` multiple times to load checks from separate files:

```python
scan.add_sodacl_yaml_file("checks/orders_checks.yml")
scan.add_sodacl_yaml_file("checks/customers_checks.yml")
```

**Inline checks** — you can also add checks as a YAML string:

```python
scan.add_sodacl_yaml_str("""
checks for orders:
  - row_count > 0
""")
```

In [ ]:
# Demonstrate inline checks (no file needed) and multi-table scanning

# Add a second table to our DuckDB file
conn2 = duckdb.connect(db_path)
conn2.execute("""
    CREATE TABLE IF NOT EXISTS customers AS
    SELECT * FROM (VALUES
        (1, 'Alice',   'alice@example.com'),
        (2, 'Bob',     'bob@example.com'),
        (3, 'Charlie', 'charlie@example.com')
    ) t(customer_id, name, email)
""")
conn2.close()

# Run a scan using inline YAML strings instead of files
scan3 = Scan()
scan3.set_data_source_name("warehouse")
scan3.add_configuration_yaml_file(str(config_path))

# Add checks for two tables as inline strings
scan3.add_sodacl_yaml_str("""
checks for orders:
  - row_count > 0
  - missing_count(amount) = 0
""")
scan3.add_sodacl_yaml_str("""
checks for customers:
  - row_count > 0
  - missing_count(email) = 0
""")

scan3.execute()

print("Multi-table inline scan results:")
print(f"  FAILs: {scan3.has_check_fails()}")
print(f"  WARNs: {scan3.has_check_warns()}")
print()
print(scan3.get_logs_text())

### What just happened?
- `add_sodacl_yaml_str()` accepts raw YAML text — useful when checks are generated programmatically.
- Calling `add_sodacl_yaml_str` multiple times **accumulates checks** — all run in a single `execute()` call.
- Soda issues **one SQL query per metric per table** — it does not load data into Python memory.
- Both tables (`orders` and `customers`) are scanned in one pass, and results appear together in the log.

## Step 7 · Triggering Failures — Verifying Checks Catch Real Problems

Checks only prove their worth when you see them FAIL on bad data. Let's inject a problem and confirm Soda catches it.

In [ ]:
# Create a second DuckDB file with intentionally bad data
bad_db_path = str(tmpdir / "bad_warehouse.duckdb")
conn_bad = duckdb.connect(bad_db_path)
conn_bad.execute("""
    CREATE TABLE orders AS
    SELECT * FROM (VALUES
        (1,    'Alice',   120.50, '2024-01-01'),
        (1,    'Bob',     250.00, '2024-01-02'),   -- duplicate order_id!
        (NULL, 'Charlie', 89.99,  '2024-01-03'),   -- NULL order_id!
        (4,    NULL,      340.00, '2024-01-04')    -- NULL customer!
    ) t(order_id, customer, amount, order_date)
""")
conn_bad.close()

# Configuration for bad data source
bad_config_yml = f"""
data_sources:
  bad_warehouse:
    type: duckdb
    path: "{bad_db_path}"
"""
bad_config_path = tmpdir / "bad_configuration.yml"
bad_config_path.write_text(bad_config_yml)

# Run checks that should FAIL
scan_bad = Scan()
scan_bad.set_data_source_name("bad_warehouse")
scan_bad.add_configuration_yaml_file(str(bad_config_path))
scan_bad.add_sodacl_yaml_str("""
checks for orders:
  - row_count > 0
  - missing_count(order_id) = 0
  - missing_count(customer) = 0
  - duplicate_count(order_id) = 0
""")
scan_bad.execute()

print("=== Bad data scan results ===")
print(f"  FAILs: {scan_bad.has_check_fails()}")
print(f"  WARNs: {scan_bad.has_check_warns()}")

if scan_bad.has_check_fails():
    print("\nFailed checks:")
    for chk in scan_bad.get_checks_fail():
        print(f"  FAIL  {chk}")

print("\n--- Full log ---")
print(scan_bad.get_logs_text())

### What just happened?
- `duplicate_count(order_id) = 0` FAILed because `order_id = 1` appears twice.
- `missing_count(order_id) = 0` FAILed because one row has `NULL` in the `order_id` column.
- `missing_count(customer) = 0` FAILed because one row has `NULL` in the `customer` column.
- **`row_count > 0` still PASSes** — the table exists and has rows, which is all that check tests.

In [ ]:
# Challenge: Add a warn/fail threshold to the orders table
#
# Task:
#   1. Create a fresh DuckDB file with an 'invoices' table that has 8 rows.
#   2. Write a checks.yml (as a string) with these checks on 'invoices':
#      a. row_count with warn: when < 5  AND  fail: when < 2
#      b. missing_count(invoice_id) = 0
#      c. duplicate_count(invoice_id) = 0
#   3. Run the scan and print whether any check FAILed or WARNed.
#
# Expected: all checks PASS (8 rows > warn threshold of 5)

# Your solution here
challenge_db_path = str(tmpdir / "challenge.duckdb")

# TODO: create invoices table with 8 rows

# TODO: write configuration and checks YAML strings

# TODO: run the scan and print results

---
## Day 1 key concepts recap

| Concept | What to remember |
|---|---|
| Soda Core | Open-source data quality framework — YAML checks, Python API, 20+ connectors |
| SodaCL | YAML dialect for checks; `checks for <table>:` blocks |
| Check anatomy | `metric(column) operator threshold` — e.g. `missing_count(id) = 0` |
| Warn vs Fail | `warn: when < N` emits WARNING; `fail: when < N` emits FAIL |
| `Scan()` API | `set_data_source_name → add_configuration → add_sodacl → execute()` |
| Results API | `has_check_fails()`, `has_check_warns()`, `get_checks_fail()`, `get_logs_text()` |
| Pushdown SQL | Soda generates SQL and runs it on the data source — no data movement |
| DuckDB connector | `soda-core-duckdb`; use a file path (not `:memory:`) for Soda scans |

> **Tip:** Always close your DuckDB connection before running a Soda scan — DuckDB only allows one write connection at a time and Soda opens its own.

---
## What's next
**Day 2** → Dive into data source configuration — DuckDB, PostgreSQL, and Snowflake connectors, environment variable substitution for credentials, and testing connectivity.

Mark Day 1 complete in your [tracker](../index.html).